# 3. Model Training

This notebook trains an ensemble of 4 models for bird audio classification. We'll use pre-trained models from Hugging Face Transformers and fine-tune them for our specific task.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('/home/kwierman/Desktop/Projects/BirdClef')
PROCESSED_DIR = BASE_DIR / 'notebooks' / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'notebooks' / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 3.1 Load Configuration and Data

In [ ]:
# Load taxonomy and mappings
taxonomy_df = pd.read_csv(BASE_DIR / 'taxonomy.csv')
species_list = taxonomy_df['primary_label'].tolist()
num_classes = len(species_list)
species_to_idx = {s: i for i, s in enumerate(species_list)}
idx_to_species = {i: s for s, i in species_to_idx.items()}

# Load processed data
train_data = pd.read_csv(PROCESSED_DIR / 'train_split.csv')
val_data = pd.read_csv(PROCESSED_DIR / 'val_split.csv')

print(f"Number of classes: {num_classes}")
print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

## 3.2 Check for Transformers

In [ ]:
try:
    from transformers import AutoModel, AutoConfig, AutoFeatureExtractor
    import librosa
    print("Transformers and librosa available")
    TRANSFORMERS_AVAILABLE = True
except ImportError as e:
    print(f"Transformers/librosa not available: {e}")
    TRANSFORMERS_AVAILABLE = False

try:
    import torchaudio
    print("torchaudio available")
    TORCHAUDIO_AVAILABLE = True
except:
    TORCHAUDIO_AVAILABLE = False

print(f"\nTransformers: {TRANSFORMERS_AVAILABLE}, torchaudio: {TORCHAUDIO_AVAILABLE}")

## 3.3 Audio Loading Function

In [ ]:
def load_audio(file_path, target_sr=32000):
    """Load audio file and resample."""
    if TORCHAUDIO_AVAILABLE:
        waveform, sr = torchaudio.load(file_path)
        if sr != target_sr:
            resampler = torchaudio.transforms.Resample(sr, target_sr)
            waveform = resampler(waveform)
        return waveform
    return None

## 3.4 Dataset Class for Audio Classification

In [ ]:
class BirdAudioDataset(Dataset):
    """Dataset for bird audio classification with pre-trained model features."""
    
    def __init__(self, dataframe, max_duration=5.0, sample_rate=32000, augment=False):
        self.dataframe = dataframe.reset_index(drop=True)
        self.max_duration = max_duration
        self.sample_rate = sample_rate
        self.augment = augment
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        
        try:
            waveform = load_audio(row['file_path'], self.sample_rate)
            
            if waveform is None:
                raise ValueError("Failed to load audio")
            
            # Trim or pad
            max_frames = int(self.max_duration * self.sample_rate)
            if waveform.shape[1] > max_frames:
                waveform = waveform[:, :max_frames]
            elif waveform.shape[1] < max_frames:
                padding = max_frames - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, padding))
            
            # Convert to mono
            if waveform.shape[0] > 1:
                waveform = waveform.mean(dim=0, keepdim=True)
            
            # Create label
            label = torch.zeros(num_classes)
            if 'label_idx' in row:
                label[int(row['label_idx'])] = 1.0
            
            return waveform.squeeze(0), label
            
        except Exception as e:
            # Return zero tensor on error
            waveform = torch.zeros(self.max_duration * self.sample_rate)
            label = torch.zeros(num_classes)
            return waveform, label

# Create datasets
train_dataset = BirdAudioDataset(train_data, augment=True)
val_dataset = BirdAudioDataset(val_data, augment=False)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

## 3.5 Model Definitions - Ensemble of 4 Models

In [ ]:
class BirdClassifier(nn.Module):
    """Base bird classifier with backbone and classification head."""
    
    def __init__(self, backbone, num_classes, hidden_size=512):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)

In [ ]:
# Model 1: CNN-based feature extractor (custom)
class CNNFeatureExtractor(nn.Module):
    """CNN-based audio feature extractor."""
    
    def __init__(self, sample_rate=32000):
        super().__init__()
        
        # Mel spectrogram conversion
        self.mel_spec = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=512, stride=160, padding=0),  # Approximate STFT
            nn.ReLU(),
        )
        
        self.conv_layers = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
    
    def forward(self, x):
        # x: (batch, time_steps)
        x = x.unsqueeze(1)  # Add channel dimension
        x = self.mel_spec(x)
        x = self.conv_layers(x)
        return x.squeeze(-1)  # (batch, 256)


class CNNModel(nn.Module):
    """CNN-based bird classifier."""
    
    def __init__(self, num_classes):
        super().__init__()
        self.features = CNNFeatureExtractor()
        self.classifier = nn.Sequential(
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        features = self.features(x)
        return self.classifier(features)

print("CNN Model defined")

In [ ]:
# Model 2: LSTM-based model
class LSTMModel(nn.Module):
    """LSTM-based bird classifier."""
    
    def __init__(self, num_classes, hidden_size=256, num_layers=2):
        super().__init__()
        
        # Simple feature projection
        self.input_proj = nn.Linear(128, 64)  # Expect mel spectrogram features
        
        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        # x: (batch, time) - raw waveform
        # Apply mel spectrogram via convolution
        x = torch.stft(x, n_fft=2048, hop_length=512, return_complex=True)
        x = torch.abs(x).transpose(1, 2)  # (batch, time, freq)
        
        # Take mel bands (first 128)
        x = x[:, :, :128]
        
        # Project
        x = self.input_proj(x)
        
        # LSTM
        lstm_out, (h_n, _) = self.lstm(x)
        features = h_n[-1]  # Last hidden state
        
        return self.classifier(features)

print("LSTM Model defined")

In [ ]:
# Model 3: ResNet-based model for audio
class ResidualBlock(nn.Module):
    """Residual block for audio features."""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm1d(out_channels)
            )
    
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return torch.relu(out)

class ResNet1D(nn.Module):
    """1D ResNet for audio classification."""
    
    def __init__(self, num_classes):
        super().__init__()
        
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm1d(64)
        self.pool = nn.MaxPool1d(3, stride=2, padding=1)
        
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(512, num_classes)
    
    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = [ResidualBlock(in_channels, out_channels, stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = x.unsqueeze(1)  # (batch, 1, time)
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

print("ResNet1D Model defined")

In [ ]:
# Model 4: Transformer-based model
class PositionalEncoding(nn.Module):
    """Positional encoding for transformer."""
    
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerModel(nn.Module):
    """Transformer-based bird classifier."""
    
    def __init__(self, num_classes, d_model=256, nhead=8, num_layers=4):
        super().__init__()
        
        self.input_proj = nn.Linear(128, d_model)  # Mel spectrogram features
        self.pos_encoder = PositionalEncoding(d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        # Convert to spectrogram
        x = torch.stft(x, n_fft=2048, hop_length=512, return_complex=True)
        x = torch.abs(x).transpose(1, 2)[:, :, :128]  # (batch, time, mel)
        
        # Project to d_model
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        
        # Transformer
        x = self.transformer_encoder(x)
        
        # Global average pooling
        x = x.mean(dim=1)
        
        return self.classifier(x)

print("Transformer Model defined")

## 3.6 Training Functions

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device, scheduler=None):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    num_batches = 0
    
    for batch_idx, (waveforms, labels) in enumerate(dataloader):
        waveforms = waveforms.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(waveforms)
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx}/{len(dataloader)}, Loss: {loss.item():.4f}")
    
    if scheduler is not None:
        scheduler.step()
    
    return total_loss / num_batches


def evaluate(model, dataloader, criterion, device):
    """Evaluate model."""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for waveforms, labels in dataloader:
            waveforms = waveforms.to(device)
            labels = labels.to(device)
            
            outputs = model(waveforms)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            # Get predictions (sigmoid for multi-label)
            preds = torch.sigmoid(outputs)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    
    # Calculate metrics
    from sklearn.metrics import accuracy_score, f1_score
    
    # Convert to binary predictions
    binary_preds = (all_preds > 0.5).astype(int)
    
    # Sample-level accuracy
    sample_acc = (binary_preds == all_labels).all(axis=1).mean()
    
    # Label-level F1
    label_f1 = f1_score(all_labels, binary_preds, average='macro', zero_division=0)
    
    return total_loss / len(dataloader), sample_acc, label_f1

print("Training functions defined")

## 3.7 Train Each Model

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

# Create data loaders
BATCH_SIZE = 32
NUM_WORKERS = 4

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
# Training configuration
EPOCHS = 10
LEARNING_RATE = 1e-3

models_config = [
    ('cnn', CNNModel(num_classes)),
    ('lstm', LSTMModel(num_classes)),
    ('resnet', ResNet1D(num_classes)),
    ('transformer', TransformerModel(num_classes))
]

trained_models = {}
training_results = []

for model_name, model in models_config:
    print(f"\n{'='*60}")
    print(f"Training {model_name.upper()} Model")
    print('='*60)
    
    model = model.to(device)
    
    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    best_val_loss = float('inf')
    best_val_f1 = 0
    patience = 3
    no_improve = 0
    
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        
        train_loss = train_epoch(model, train_loader, criterion, optimizer, device, scheduler)
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion, device)
        
        print(f"  Train Loss: {train_loss:.4f}")
        print(f"  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_val_loss = val_loss
            torch.save(model.state_dict(), MODELS_DIR / f'{model_name}_best.pt')
            no_improve = 0
            print(f"  -> Saved best model (F1: {val_f1:.4f})")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break
    
    # Load best model
    model.load_state_dict(torch.load(MODELS_DIR / f'{model_name}_best.pt'))
    trained_models[model_name] = model
    
    training_results.append({
        'model': model_name,
        'best_val_loss': best_val_loss,
        'best_val_f1': best_val_f1
    })
    
    print(f"\n{model_name} training complete. Best F1: {best_val_f1:.4f}")

## 3.8 Ensemble Predictions

In [ ]:
def ensemble_predict(models, dataloader, device):
    """Make ensemble predictions."""
    all_preds = []
    
    for model in models.values():
        model.eval()
        model_preds = []
        
        with torch.no_grad():
            for waveforms, _ in dataloader:
                waveforms = waveforms.to(device)
                outputs = model(waveforms)
                preds = torch.sigmoid(outputs)
                model_preds.append(preds.cpu().numpy())
        
        all_preds.append(np.vstack(model_preds))
    
    # Average predictions
    ensemble_preds = np.mean(all_preds, axis=0)
    return ensemble_preds

# Evaluate ensemble on validation set
ensemble_preds = ensemble_predict(trained_models, val_loader, device)

# Get ground truth labels
val_labels = []
for _, labels in val_loader:
    val_labels.append(labels.numpy())
val_labels = np.vstack(val_labels)

# Calculate ensemble metrics
from sklearn.metrics import f1_score, accuracy_score

binary_preds = (ensemble_preds > 0.5).astype(int)
ensemble_f1 = f1_score(val_labels, binary_preds, average='macro')
ensemble_acc = (binary_preds == val_labels).all(axis=1).mean()

print(f"Ensemble Results:")
print(f"  Macro F1: {ensemble_f1:.4f}")
print(f"  Exact Match Accuracy: {ensemble_acc:.4f}")

## 3.9 Save Ensemble Models

In [ ]:
# Save model configuration
ensemble_config = {
    'num_classes': num_classes,
    'species_list': species_list,
    'training_results': training_results
}

import json
with open(MODELS_DIR / 'ensemble_config.json', 'w') as f:
    json.dump(ensemble_config, f, indent=2)

# Summary
print("\n" + "="*60)
print("ENSEMBLE TRAINING SUMMARY")
print("="*60)

for result in training_results:
    print(f"\n{result['model']}:")
    print(f"  Best Val Loss: {result['best_val_loss']:.4f}")
    print(f"  Best Val F1: {result['best_val_f1']:.4f}")

print(f"\nEnsemble Macro F1: {ensemble_f1:.4f}")
print(f"Ensemble Exact Match Acc: {ensemble_acc:.4f}")

print(f"\nModels saved to: {MODELS_DIR}")
print("\n✅ Model training complete!")